# Capstone — Which pages should an editor review first?

**Lane 2 — Refresh / Content Opportunity Scoring.** This notebook mirrors my deployed paper (link in `submission/paper_url.txt`). Every number on the paper is written by section 7 of this notebook.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/WingManOO7/ML-INTER/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. Setup

The same shared module and the same ten development decision dates as ML-09 and ML-10. June 2026 is
opened **once**, in the sealed cell of section 4b, after everything above it is frozen.

In [1]:
import os, sys, json, subprocess, importlib.util, urllib.request, warnings
from pathlib import Path
warnings.filterwarnings('ignore', category=FutureWarning)

if importlib.util.find_spec('duckdb') is None:      # Colab ships duckdb; this covers fresh local envs
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'duckdb'])

import numpy as np
import pandas as pd
import sklearn

# the shared module: from the repo when run inside it, else fetched from the repo (Colab opened from GitHub)
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'work' / 'notebooks').is_dir()), None)
if ROOT is not None:
    sys.path.insert(0, str(ROOT / 'work' / 'scripts'))
else:
    ROOT = Path.cwd()
    urllib.request.urlretrieve('https://raw.githubusercontent.com/WingManOO7/ML-INTER/main/work/scripts/search_queue.py',
                               'search_queue.py')
import search_queue as sq

# Token order: env var -> Colab Secret -> prompt (last resort). Never paste a token into a cell.
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
if not HF_TOKEN:
    import getpass
    HF_TOKEN = getpass.getpass('Hugging Face READ token: ')

OUT_DIR = ROOT / 'work' / 'outputs'
CACHE = OUT_DIR / 'cache'            # parquet frames, gitignored: reruns skip the warehouse scans
OUT_DIR.mkdir(parents=True, exist_ok=True)

DEV_DATES = sq.decision_dates('2026-01-05', '2026-05-11')     # every second Monday; outcomes end 2026-05-24
DEV_MONTHS = sq.months_needed(DEV_DATES)
assert '2026-06' not in DEV_MONTHS, 'June 2026 is the sealed month: development never opens it'

con = sq.open_warehouse(DEV_MONTHS, HF_TOKEN)
frames = sq.build_frames(con, DEV_DATES, cache_dir=CACHE)
dev = frames[frames['label_ok']].reset_index(drop=True)
SEED = 42

print(f'{len(DEV_DATES)} decision dates {DEV_DATES[0].date()} -> {DEV_DATES[-1].date()} | months {DEV_MONTHS[0]} -> {DEV_MONTHS[-1]}')
print(f'{len(frames):,} page-date rows; {len(dev):,} with a complete outcome window; '
      f'{dev.client_hash_id.nunique()} clients | pandas {pd.__version__}, scikit-learn {sklearn.__version__}')

import hashlib, time
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
make_lr = lambda: make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
make_hgb = lambda: HistGradientBoostingClassifier(learning_rate=0.05, max_iter=300, random_state=SEED)
FEATURES_8 = sq.CONTRACT_FEATURES + sq.EXTRA_FEATURES

10 decision dates 2026-01-05 -> 2026-05-11 | months 2025-12 -> 2026-05
659,482 page-date rows; 654,186 with a complete outcome window; 50 clients | pandas 2.3.3, scikit-learn 1.7.2


## 1. Question

*The research question and the decision it supports.*

**The question.** At each decision date, which **50 pages** should an editor review first, because their
search visibility is dropping in a way that is likely to *last* and large enough to *matter*?

- **Decision supported:** how to spend a fixed review budget: about 50 pages per two-week cycle, the
  capacity fixed in ML-03.
- **Unit:** one page at one decision date T. Features come from the 14 days before T; the outcome from the 14
  days after.
- **Output:** a ranked queue. Every row carries an **action** (refresh review, technical check, site-level
  check, monitor), a **reason code** and a **confidence band**.
- **Cost of a wrong call:** there are three ways to fail, not two.
  - **Wasted hour:** the page was fine.
  - **Missed loss:** a decaying page never surfaced.
  - **Right but irrelevant:** the page did decline, but it was too small for the hour to matter.
    ML-08 found this third failure, which is why every result below reports **precision and visibility
    together**.
- **Why data and ML help:** about 40–80 thousand eligible pages per cycle compete for 50 slots. A rule that
  ranks by "biggest drop" is easy to state and reasonably good (ML-07). The question is whether a learned
  probability makes the same budget land on more real, lasting losses.

In [2]:
per_cycle = frames.groupby('decision_date').size()
print(f'eligible pages per cycle: {per_cycle.min():,} to {per_cycle.max():,}; a 50-page queue reviews '
      f'{50 / per_cycle.max():.2%} to {50 / per_cycle.min():.2%} of them')

eligible pages per cycle: 42,331 to 82,673; a 50-page queue reviews 0.06% to 0.12% of them


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

- **Release:** `FlyRank/internship-warehouse`, build v20260703: a pseudonymized warehouse of
  78,835,655 daily search rows (2025-01-27 → 2026-06-30), gated on Hugging Face.
- **Tables used:**
  - `fact_content_daily_performance`, month partitions 2025-12 → 2026-05 for development and 2026-04 →
    2026-06 for the sealed test. Search Console columns only, every row filtered with
    `gsc_data_available IS TRUE`.
  - `dim_content`, for one column only: `content_created_date`.
- **Decision dates:**
  - Development: every second Monday, **2026-01-05 → 2026-05-11** (10 dates; the last outcome window ends
    2026-05-24).
  - Sealed: **2026-05-25 and 2026-06-08**, whose outcome windows fall in June.
- **Eligibility:** ≥ 100 impressions in the 14 days before T, tracked from the window's first day, a valid
  position, and, for anything that needs a label, all 14 outcome days tracked.

**Excluded, and why:**

| Excluded | Why |
|---|---|
| `content_updated_date`, `last_optimized_date`, `is_published`, `is_deleted` | July snapshot: the update date lies after T for 76–100% of pages (ML-09) |
| GA4, AI-referral and scroll columns | outside this search question; GA4 availability is FALSE or NULL on most rows (ML-04) |
| `search_volume`, `cpc`, `competition` | keyword-tool values with no date; I can't show they were known at T |
| `gsc_clicks` | not needed for a visibility label; CTR is Lane 4 |
| all ID columns as features | pseudonyms: grouping and splitting only; ML-09 showed what an ID feature does to a careless split |
| pages below the floor, or with an incomplete outcome window | too small for a 20% change to mean anything; a tracking gap would look like a decline |

**Public-safe:** only aggregates and charts leave this notebook. The queue CSVs carry pseudonymous IDs
and are gitignored. No client names, URLs, queries or titles exist in the release or in any output.

In [3]:
data_table = frames.groupby('decision_date').agg(
    pages=('content_hash_id', 'size'), clients=('client_hash_id', 'nunique'),
    complete_outcome=('label_ok', 'mean')).join(
    dev.groupby('decision_date').agg(base_rate_robust=('declined_robust', 'mean'), base_rate_contract=('declined', 'mean')))
print(f"{len(frames):,} page-date rows ({frames.content_hash_id.nunique():,} distinct pages, {frames.client_hash_id.nunique()} clients); "
      f"{(~frames.label_ok).sum():,} rows ({(~frames.label_ok).mean():.1%}) dropped for an incomplete outcome window\n")
data_table.round(3)

659,482 page-date rows (118,815 distinct pages, 51 clients); 5,296 rows (0.8%) dropped for an incomplete outcome window



,pages,clients,complete_outcome,base_rate_robust,base_rate_contract
decision_date,,,,,
2026-01-05,42331,27,0.993,0.253,0.252
2026-01-19,45420,27,0.996,0.136,0.139
2026-02-02,53419,28,0.993,0.215,0.216
2026-02-16,57263,28,0.948,0.178,0.176
2026-03-02,59716,28,0.976,0.307,0.298
2026-03-16,74542,36,1.000,0.333,0.337
2026-03-30,82593,37,1.000,0.394,0.393
2026-04-13,81947,41,1.000,0.425,0.417
2026-04-27,82673,45,1.000,0.572,0.578


**What the data looks like.**
- **Volume:** 659,482 page-date rows, covering 118,815 distinct pages from 51 clients over ten dates. 5,296 rows (0.8%) are dropped because their outcome window wasn't fully tracked; almost all sit in January to early March, while the panel was still filling in.
- **Growth:** the panel grows from 42,331 eligible pages (27 clients) on 2026-01-05 to about 80,000 (45 clients) by April.
- **The base rate is not stable.** It runs from **0.14** (2026-01-19) to **0.57** (2026-04-27), when the whole portfolio lost 18% of its impressions over the next two weeks. A fixed −20% threshold partly measures the market, so every result below is read against **its own date's base rate**.
- **The two labels agree closely:** the robust and contract labels are within 0.01 of each other on every date.

## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

**Assumptions.**
- An editor reviews about 50 pages per two-week cycle.
- A drop that persists for two weeks is worth a look.
- Page-level decisions are made against the page's own recent level.

**Features (five, all from before T; ML-04 contract):**

| Feature | Known at T because… |
|---|---|
| `log_imp_pre`: log impressions, 14 days before T | every day summed ends before T |
| `pos_pre`: impression-weighted position, skipping position 0 | positions already reported before T |
| `active_days_pre`: days with impressions (of 14) | counted before T |
| `momentum_pre`: log(week 2 / week 1 impressions) | compares the past with the past |
| `age_days_at_T`: T − created date | the creation date precedes T and does not move |

Three more pre-T candidates (`momentum_4w`, `peak_day_share_pre`, `site_momentum`) are tested and have to
earn their place.

**Label — `declined_robust`.** Impressions over the 14 days from T fall more than **20%** below the 14 days
before, after dropping **each window's single biggest day**. ML-07 found one-day spikes fooling the plain
label. The ML-04 contract label `declined` is kept as a robustness check.

**Baseline — the frozen ML-07 rule.** Flag a page when week-2 impressions are more than 20% below week 1,
and rank by impressions lost. If a page simply stays at its week-2 level, its two-week loss is exactly that
number, so **the rule is a persistence forecast of the loss**.

**The policy — score = P(decline) × lost weekly impressions.** The rule's loss, weighted by the chance it
persists. P comes from logistic regression on the five features. A learned probability alone ranks tiny
collapsed pages first (ML-08), so the score has to carry the stake.

**Selection rule (declared before the backtest).**
- Metrics: **precision@50** is primary (fixed in ML-03). **Visibility@50** is a guardrail, added after the
  ML-08 error analysis and before the sealed test: the share of the largest achievable lost impressions
  that the 50 picks actually cover.
- A candidate must beat the rule on precision@50 **without** losing visibility@50. Among those that pass,
  the simplest wins.

**Validation.**
- **Rolling-origin × client-grouped.** At each of 8 test dates (Feb 2 → May 11), models train only on
  *earlier* dates and on clients in *other* folds, then score the held-out clients: 8 dates × 5 folds =
  **40 queues**.
- **Uncertainty:** paired differences get 95% intervals from a bootstrap that resamples **whole dates**
  (folds of one date share a market week).
- **Sealed test:** June, opened once (§4b).

**Leakage controls.**
- 21 unit tests, re-run below.
- An outcome-feature guard in the module.
- A planted-leak test and a shuffled-label test (ML-09).
- The snapshot columns are excluded.
- Product flags don't exist in the release.

In [4]:
tests = ROOT / 'work' / 'tests'
if tests.is_dir() and importlib.util.find_spec('pytest'):
    run = subprocess.run([sys.executable, '-m', 'pytest', str(tests), '-q', '-p', 'no:cacheprovider', '--color=no'],
                         capture_output=True, text=True, cwd=ROOT)
    print('test suite:', run.stdout.strip().splitlines()[-1])
    assert run.returncode == 0, run.stdout
else:
    print('test suite: not available here (run `python -m pytest work/tests` from a clone)')
sq.assert_no_outcome_features(FEATURES_8)
print('features:', sq.CONTRACT_FEATURES, '| extra candidates:', sq.EXTRA_FEATURES)
print('routes:', [(a, r) for a, r, _ in sq.ROUTES])

test suite: 21 passed in 8.26s
features: ['log_imp_pre', 'pos_pre', 'active_days_pre', 'momentum_pre', 'age_days_at_T'] | extra candidates: ['momentum_4w', 'peak_day_share_pre', 'site_momentum']
routes: [('TECHNICAL_CHECK', 'COLLAPSE_TO_NEAR_ZERO'), ('SITE_LEVEL_CHECK', 'SITE_WIDE_SLIDE'), ('MONITOR', 'SPIKE_FADE'), ('MONITOR', 'NEW_PAGE_SETTLING')]


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

### 4a. Development: 40 rolling-origin, client-grouped queues

Every policy is scored on the **same 40 queues**, computed in this run.

In [5]:
t0 = time.time()
bt = sq.rolling_backtest(frames, {'lr5': (sq.CONTRACT_FEATURES, make_lr), 'lr8': (FEATURES_8, make_lr),
                                  'hgb8': (FEATURES_8, make_hgb)})
rng = np.random.RandomState(SEED)
bt['random'] = rng.rand(len(bt))
POLICIES = {
    'random order': 'random',
    'ML-07 rule (baseline)': sq.rule_score,
    'slide sort (no training)': lambda c: -c['momentum_pre'],
    'LR probability alone': 'p_lr5',
    'LR x impressions': lambda c: c['p_lr5'] * c['imp_pre'],
    'HGB (8 features) x lost impressions': lambda c: sq.queue_score(c['p_hgb8'], c),
    'LR (8 features) x lost impressions': lambda c: sq.queue_score(c['p_lr8'], c),
    'LR x lost impressions (policy)': lambda c: sq.queue_score(c['p_lr5'], c),
}
cells_robust = sq.cell_metrics(bt, POLICIES)
ref = cells_robust[cells_robust['policy'] == 'ML-07 rule (baseline)'].set_index(['decision_date', 'fold'])

def summarise(cells):
    out = {}
    for name, g in cells.groupby('policy', sort=False):
        g = g.set_index(['decision_date', 'fold'])
        out[name] = {'P@50': g['P@50'].mean(), 'P@50 sd': g['P@50'].std(), 'lift': g['lift@50'].mean(),
                     'visibility@50': g['visibility@50'].mean(), 'visibility sd': g['visibility@50'].std(),
                     'AP': g['AP'].mean(), 'AUC': g['AUC'].mean(),
                     'P >= rule': int((g['P@50'] >= ref.loc[g.index, 'P@50']).sum()),
                     'vis >= rule': int((g['visibility@50'] >= ref.loc[g.index, 'visibility@50']).sum())}
    return pd.DataFrame(out).T

results = summarise(cells_robust)
passes = results[(results['P@50'] > results.loc['ML-07 rule (baseline)', 'P@50']) &
                 (results['visibility@50'] >= results.loc['ML-07 rule (baseline)', 'visibility@50'])]
print(f"backtest: {cells_robust.groupby(['decision_date', 'fold']).ngroups} queues, {len(bt):,} scored page-dates, {time.time() - t0:.0f}s")
print('pass the selection rule (beat the rule on P@50 without losing visibility):', list(passes.index), '\n')
results.round(3)

backtest: 40 queues, 566,918 scored page-dates, 419s
pass the selection rule (beat the rule on P@50 without losing visibility): ['HGB (8 features) x lost impressions', 'LR (8 features) x lost impressions', 'LR x lost impressions (policy)'] 



,P@50,P@50 sd,lift,visibility@50,visibility sd,AP,AUC,P >= rule,vis >= rule
random order,0.389,0.175,1.006,0.040,0.033,0.383,0.499,1.0,0.0
ML-07 rule (baseline),0.640,0.180,1.919,0.412,0.147,0.490,0.599,40.0,40.0
slide sort (no training),0.827,0.135,2.513,0.090,0.119,0.556,0.665,38.0,0.0
LR probability alone,0.810,0.154,2.443,0.077,0.115,0.543,0.653,33.0,0.0
LR x impressions,0.412,0.151,1.229,0.627,0.132,0.401,0.530,4.0,38.0
HGB (8 features) x lost impressions,0.686,0.179,2.075,0.430,0.162,0.502,0.640,34.0,19.0
LR (8 features) x lost impressions,0.702,0.170,2.134,0.417,0.158,0.503,0.642,34.0,12.0
LR x lost impressions (policy),0.694,0.171,2.106,0.421,0.156,0.501,0.641,34.0,16.0


In [6]:
POLICY = 'LR x lost impressions (policy)'
boot = {m: sq.date_bootstrap(cells_robust, POLICY, 'ML-07 rule (baseline)', m) for m in ['P@50', 'visibility@50', 'AUC']}
print('Policy minus frozen rule — mean paired difference [95% interval, whole dates resampled]:')
for m, (d, lo, hi) in boot.items():
    print(f'  {m:<14} {d:+.3f}  [{lo:+.3f}, {hi:+.3f}]')

per_date = cells_robust.pivot_table(index='decision_date', columns='policy', values=['P@50', 'base_rate'])
per_date = pd.DataFrame({'base rate': per_date[('base_rate', POLICY)],
                         'rule P@50': per_date[('P@50', 'ML-07 rule (baseline)')],
                         'policy P@50': per_date[('P@50', POLICY)],
                         'slide sort P@50': per_date[('P@50', 'slide sort (no training)')]})
per_date['policy >= rule'] = per_date['policy P@50'] >= per_date['rule P@50']
print(); print(per_date.round(3).to_string(), '\n')

# error analysis promoted to policy: how the policy's top-50 picks behave per route
bt['policy_score'] = sq.queue_score(bt['p_lr5'], bt)
picks = bt[bt.groupby(['decision_date', 'fold'])['policy_score'].rank(ascending=False, method='first') <= 50]
picks = picks.join(sq.route(picks))
route_table = picks.groupby(['action', 'reason_code']).agg(picks=('declined_robust', 'size'), precision=('declined_robust', 'mean'),
                                                           drop_share=('gap_robust', 'sum'))
route_table['share_of_picks'] = route_table['picks'] / len(picks)
route_table['drop_share'] = route_table['drop_share'] / picks['gap_robust'].sum()
route_table.sort_values('picks', ascending=False).round(3)

Policy minus frozen rule — mean paired difference [95% interval, whole dates resampled]:
  P@50           +0.054  [+0.030, +0.079]
  visibility@50  +0.009  [-0.031, +0.056]
  AUC            +0.042  [+0.026, +0.059]

               base rate  rule P@50  policy P@50  slide sort P@50  policy >= rule
decision_date                                                                    
2026-02-02         0.400      0.792        0.796            0.892            True
2026-02-16         0.227      0.552        0.664            0.744            True
2026-03-02         0.256      0.528        0.560            0.644            True
2026-03-16         0.345      0.452        0.544            0.896            True
2026-03-30         0.430      0.712        0.756            0.848            True
2026-04-13         0.462      0.692        0.712            0.928            True
2026-04-27         0.567      0.748        0.832            0.904            True
2026-05-11         0.375      0.640        0.6

picks  precision  drop_share  \
action           reason_code                                           
REFRESH_REVIEW   SLIDING_WITH_DEMAND     1285      0.633       0.562   
SITE_LEVEL_CHECK SITE_WIDE_SLIDE          347      0.795       0.325   
TECHNICAL_CHECK  COLLAPSE_TO_NEAR_ZERO    190      0.900       0.079   
MONITOR          NEW_PAGE_SETTLING        111      0.685       0.021   
                 SPIKE_FADE                67      0.746       0.014   

                                        share_of_picks  
action           reason_code                            
REFRESH_REVIEW   SLIDING_WITH_DEMAND             0.642  
SITE_LEVEL_CHECK SITE_WIDE_SLIDE                 0.174  
TECHNICAL_CHECK  COLLAPSE_TO_NEAR_ZERO           0.095  
MONITOR          NEW_PAGE_SETTLING               0.056  
                 SPIKE_FADE                      0.034

In [7]:
# robustness: the same comparison on the ML-04 contract label (no spike trimming), LR refit on that label
bt_contract = sq.rolling_backtest(frames, {'lr5': (sq.CONTRACT_FEATURES, make_lr)}, label='declined')
cells_contract = sq.cell_metrics(bt_contract, {'ML-07 rule (baseline)': sq.rule_score,
                                               'slide sort (no training)': lambda c: -c['momentum_pre'],
                                               'LR x lost impressions (policy)': lambda c: sq.queue_score(c['p_lr5'], c)},
                                 label='declined', gap='gap')
contract = cells_contract.groupby('policy')[['P@50', 'visibility@50', 'AUC']].mean()
print('Contract label (declined):'); print(contract.round(3).to_string())

# the two signal checks behind the rule, on all ten dates (ML-07 ran them on one date)
slide_band = pd.cut(dev['momentum_pre'], [-np.inf, np.log(0.5), np.log(0.8), np.log(1.25), np.log(2), np.inf],
                    labels=['collapsed (<0.5x)', 'down 20-50%', 'about flat', 'up 25-100%', 'up >2x'])
age_band = pd.cut(dev['age_days_at_T'], [0, 30, 90, 180, 365, np.inf], right=False,
                  labels=['<30 d', '30-89 d', '90-179 d', '180-364 d', '365+ d'])
signals = {'slide': dev.groupby(slide_band, observed=True)['declined_robust'].agg(n='size', rate='mean'),
           'age': dev.groupby(age_band, observed=True)['declined_robust'].agg(n='size', rate='mean')}
print('\nSignal checks over all ten dates:'); [print(k, v.round(3).to_dict('index')) for k, v in signals.items()]

Contract label (declined):
                                 P@50  visibility@50    AUC
policy                                                     
LR x lost impressions (policy)  0.722          0.475  0.648
ML-07 rule (baseline)           0.663          0.463  0.608
slide sort (no training)        0.872          0.118  0.669



Signal checks over all ten dates:
slide {'collapsed (<0.5x)': {'n': 49570, 'rate': 0.717}, 'down 20-50%': {'n': 128015, 'rate': 0.471}, 'about flat': {'n': 304542, 'rate': 0.297}, 'up 25-100%': {'n': 128673, 'rate': 0.203}, 'up >2x': {'n': 43386, 'rate': 0.232}}
age {'<30 d': {'n': 37724, 'rate': 0.315}, '30-89 d': {'n': 131740, 'rate': 0.382}, '90-179 d': {'n': 153583, 'rate': 0.325}, '180-364 d': {'n': 263055, 'rate': 0.336}, '365+ d': {'n': 68084, 'rate': 0.321}}


[None, None]

**Main result (40 queues, the tables above).**
- **Only the policy family beats the rule on precision without giving up visibility.** The policy (P × lost impressions) reaches **precision@50 = 0.694, against 0.640 for the frozen rule**, with visibility@50 of 0.421 vs 0.412. The paired whole-date bootstrap gives:
  - precision: **+0.054 [+0.030, +0.079]**
  - visibility: +0.009 [−0.031, +0.056], so no measurable loss
  - AUC: +0.042 [+0.026, +0.059]

  Precision is higher or equal in **34 of 40** queues and on **all 8 dates**.
- **Precision alone would have picked the wrong system.** The learned probability alone (0.810) and a no-training slide sort (0.827) top the precision column, but they cover only **8–9%** of the achievable lost visibility: they rank small collapsed pages first. Ranking by probability × total impressions swings the other way (visibility 0.627, precision 0.412). The policy sits on the useful part of that frontier.
- **Complexity did not earn its place.** Putting gradient boosting or the three extra features inside the same score gives 0.686–0.702 precision and 0.417–0.430 visibility, within noise of the five-feature logistic version. The selection rule keeps the simplest.
- **It holds on the contract label too:** 0.722 vs 0.663 precision, and 0.475 vs 0.463 visibility.
- **The signal checks over all ten dates repeat ML-07.** The week-over-week slide is CONFIRMED: decline rate 0.717 for collapsed pages, down to 0.20–0.23 for rising ones. Page age is flat (0.315–0.382, no trend with age), so the refresh-flag staleness signal stays out.
- **Error analysis became policy.** Among the policy's top-50 picks, the routes behave differently:
  - clean `SLIDING_WITH_DEMAND` picks persist 63% of the time and carry 56% of the lost visibility;
  - collapses persist 90% of the time and site-wide slides 80%.

  The model is right that those pages decline, but the right action is a technical or site-level check, not a rewrite. That's why routes change the **action**, not the rank.

### 4b. The sealed month (June 2026) — computed once

**Protocol.** The policy, its features, routes and thresholds were frozen above. The final model is trained
on **all** development data: ten dates, all clients, every label observed by 2026-05-24. It is then scored on
the two sealed decision dates, **2026-05-25** and **2026-06-08**, whose outcome windows fall in June. This is
the first time any June data is read.

**Guard against reuse.** The cell fingerprints the frozen setup (features, label, routes, development
dates, model coefficients). On its first run it writes `work/outputs/sealed_june_receipts.json`. On every
later run it **recomputes and asserts** that both the fingerprint and the numbers are unchanged. A changed
policy can't be quietly re-scored on June, so "evaluated once" can be checked from the repo.

In [8]:
SEALED_DATES = [pd.Timestamp('2026-05-25'), pd.Timestamp('2026-06-08')]
SEALED_RECEIPT = OUT_DIR / 'sealed_june_receipts.json'

final_model = sq.fit_policy_model(dev)                     # frozen: all development data, nothing else
coef = final_model.named_steps['logisticregression']
fingerprint = hashlib.sha256(json.dumps({
    'features': sq.CONTRACT_FEATURES, 'label': 'declined_robust', 'policy': 'p_decline * lost_weekly_impressions',
    'routes': [(a, r) for a, r, _ in sq.ROUTES], 'thresholds': [sq.COLLAPSE_SHARE, float(sq.SITE_SLIDE), sq.SPIKE_SHARE, sq.NEW_PAGE_DAYS],
    'dev_dates': [str(d.date()) for d in DEV_DATES], 'coef': np.round(coef.coef_, 6).tolist(), 'intercept': np.round(coef.intercept_, 6).tolist(),
}, sort_keys=True).encode()).hexdigest()[:16]

sealed_con = sq.open_warehouse(sq.months_needed(SEALED_DATES), HF_TOKEN)   # <- the first read of June
sealed_frames = sq.build_frames(sealed_con, SEALED_DATES, cache_dir=CACHE)
sealed = sealed_frames[sealed_frames['label_ok']].reset_index(drop=True)
sq.assert_time_forward(dev, sealed)

sealed['p_lr5'] = final_model.predict_proba(sealed[sq.CONTRACT_FEATURES])[:, 1]
sealed['random'] = np.random.RandomState(SEED).rand(len(sealed))
sealed['fold'] = sealed['client_hash_id'].map(sq.client_folds(dev['client_hash_id']))
sealed['fold'] = sealed['fold'].fillna(sealed['client_hash_id'].map(lambda c: int(hashlib.md5(c.encode()).hexdigest(), 16) % 5)).astype(int)
SEALED_POLICIES = {k: POLICIES[k] for k in ['random order', 'ML-07 rule (baseline)', 'slide sort (no training)',
                                            'LR probability alone', 'LR x lost impressions (policy)']}
sealed_cells = sq.cell_metrics(sealed, SEALED_POLICIES)
sealed_table = sealed_cells.groupby('policy', sort=False)[['P@50', 'lift@50', 'visibility@50', 'AUC']].mean()
sealed_ref = sealed_cells[sealed_cells['policy'] == 'ML-07 rule (baseline)'].set_index(['decision_date', 'fold'])
pol = sealed_cells[sealed_cells['policy'] == POLICY].set_index(['decision_date', 'fold'])
sealed_by_date = sealed_cells.pivot_table(index='decision_date', columns='policy', values=['P@50', 'base_rate'])

# portfolio view: ONE 50-page queue per sealed date across all clients
portfolio = pd.DataFrame({name: {str(d.date()): sq.queue_metrics(g, np.asarray(s(g) if callable(s) else g[s], dtype=float))['P@50']
                                  for d, g in sealed.groupby('decision_date')} for name, s in SEALED_POLICIES.items()})
new_clients = sorted(set(sealed['client_hash_id']) - set(dev['client_hash_id']))

sealed_result = {
    'fingerprint': fingerprint, 'sealed_dates': [str(d.date()) for d in SEALED_DATES],
    'pages': int(len(sealed)), 'clients': int(sealed['client_hash_id'].nunique()), 'new_clients': len(new_clients),
    'base_rate_by_date': {str(pd.Timestamp(d).date()): round(float(v), 4) for d, v in sealed.groupby('decision_date')['declined_robust'].mean().items()},
    'fold_queues': int(len(pol)), 'table': sealed_table.round(4).to_dict(orient='index'),
    'policy_P_ge_rule': int((pol['P@50'] >= sealed_ref.loc[pol.index, 'P@50']).sum()),
    'policy_vis_ge_rule': int((pol['visibility@50'] >= sealed_ref.loc[pol.index, 'visibility@50']).sum()),
    'by_date_P@50': {str(pd.Timestamp(d).date()): {k: round(float(sealed_by_date.loc[d, ('P@50', k)]), 4) for k in SEALED_POLICIES}
                     for d in sealed_by_date.index},
    'portfolio_queue_P@50': portfolio.round(4).to_dict(orient='index'),
}
if SEALED_RECEIPT.exists():
    first = json.loads(SEALED_RECEIPT.read_text())
    assert first['result']['fingerprint'] == fingerprint, 'the frozen setup changed after the sealed run: June cannot be re-used'
    assert json.dumps(first['result'], sort_keys=True) == json.dumps(sealed_result, sort_keys=True), 'sealed numbers did not reproduce'
    print(f"sealed receipts reproduced exactly (first computed {first['first_computed']})")
else:
    SEALED_RECEIPT.write_text(json.dumps({'first_computed': time.strftime('%Y-%m-%d %H:%M'), 'result': sealed_result}, indent=2))
    print('sealed receipts written for the first time')
print(f"fingerprint {fingerprint} | {sealed_result['pages']:,} pages, {sealed_result['clients']} clients "
      f"({len(new_clients)} never seen in development) | base rates {sealed_result['base_rate_by_date']}\n")
print(f"policy vs rule over {len(pol)} sealed queues: P@50 higher or equal in {sealed_result['policy_P_ge_rule']}, "
      f"visibility higher or equal in {sealed_result['policy_vis_ge_rule']}")
print('one portfolio-wide queue per date, P@50:'); print(portfolio.round(3).to_string(), '\n')
sealed_table.round(3)

sealed receipts reproduced exactly (first computed 2026-10-23 05:19)
fingerprint d9e8d9bfbd813d89 | 163,200 pages, 47 clients (4 never seen in development) | base rates {'2026-05-25': 0.4846, '2026-06-08': 0.594}

policy vs rule over 10 sealed queues: P@50 higher or equal in 9, visibility higher or equal in 2
one portfolio-wide queue per date, P@50:
            random order  ML-07 rule (baseline)  slide sort (no training)  LR probability alone  LR x lost impressions (policy)
2026-05-25          0.50                   0.80                      0.94                  0.96                            0.82
2026-06-08          0.68                   0.88                      0.98                  1.00                            0.92 



,P@50,lift@50,visibility@50,AUC
policy,,,,
random order,0.514,1.009,0.025,0.499
ML-07 rule (baseline),0.810,1.671,0.560,0.624
slide sort (no training),0.906,1.871,0.040,0.673
LR probability alone,0.902,1.856,0.026,0.680
LR x lost impressions (policy),0.854,1.758,0.544,0.656


**Sealed June, computed once.** Fingerprint `d9e8d9bfbd813d89`; 163,200 pages from 47 clients, 4 of them never seen in development; 10 queues over 2 dates.
- **The precision lead replicated.** The policy reached **0.854**, against **0.810** for the rule (lift 1.76× vs 1.67×), and was higher or equal in **9 of 10** queues. With one portfolio-wide 50-page queue per date, it scored 0.82 vs 0.80 (May 25) and 0.92 vs 0.88 (June 8).
- **The visibility guardrail did not hold queue by queue.** The policy covered 0.544 against the rule's 0.560 (−0.016), and was higher or equal in only **2 of 10** queues. The gap is small next to the development spread (SD ≈ 0.15), but in June it favours the rule, so I report it as a **miss on the guardrail**, not a tie.
- **June is a different regime.** The base rate reached **0.48 and 0.59**; the second is above anything in development (max 0.57). When most pages are falling, every ranking looks better in raw precision, which is why lift over the base rate is the fairer reading.
- **The same frontier as development.** The precision-only rankers again lead on precision (0.90–0.91) and trail on visibility (2.6–4.0%).

## 5. Limitations

*What this work cannot claim.*

1. **A short history with clear regime changes.** Eight development dates (Feb–May 2026) and two sealed dates (June), with the base rate moving from 0.14 to 0.59. The results describe this period and this portfolio. A longer, multi-season backtest is the most valuable missing experiment.
2. **High variance between queues.** The policy's precision@50 ranged from 0.28 to 0.98 across the 40 development queues. The mean advantage over the rule is real (its interval excludes zero), but any single cycle can go either way.
3. **The label partly measures the market.** A fixed 20% threshold counts site-wide and portfolio-wide swings as page declines (2026-04-27). The routes and triggers mitigate this; a **site-relative label** is the next refinement.
4. **Persistence, not early warning.** The strongest signal is a drop already visible at T (ML-08 §4d). The system does not predict fresh drops for pages that look fine today.
5. **The visibility guardrail slipped in June:** 0.544 for the policy vs 0.560 for the rule, lower in 8 of 10 queues.
6. **Concentration.** The largest client holds 22–27% of eligible pages at every date, so pooled numbers lean toward a few sites. The grouped folds at least keep each site out of its own training.
7. **Search Console only, above-floor pages only.** On 2026-05-11 the eligible pages were 21% of tracked pages but held 97% of impressions. Absence from the queue is not evidence of health, and there are no clicks, engagement or conversion outcomes.
8. **No causal claim.** Nothing here shows that reviewing or refreshing a flagged page changes its outcome. That needs an experiment, for example randomising which queued pages are reviewed first.
9. **Data quality.**
   - 0.9% of queue rows with ≥ 1,000 impressions share an identical Search Console series with another record (two in the live top 50).
   - About 3% of March 2026 rows that have a position report it below 1, which is impossible in Search Console (ML-07).
   - The `dim_content` dates are export snapshots.

In [9]:
pol_cells = cells_robust[cells_robust['policy'] == POLICY]
print(f"policy precision@50 across the 40 development queues: {pol_cells['P@50'].min():.2f} to {pol_cells['P@50'].max():.2f}")
print(f"base rate across the development dates: {data_table['base_rate_robust'].min():.2f} to {data_table['base_rate_robust'].max():.2f}")
top_client_share = dev.groupby('decision_date')['client_hash_id'].agg(lambda s: s.value_counts(normalize=True).iloc[0])
print(f"largest client's share of eligible pages per date: {top_client_share.min():.0%} to {top_client_share.max():.0%}")

policy precision@50 across the 40 development queues: 0.28 to 0.98
base rate across the development dates: 0.14 to 0.57


largest client's share of eligible pages per date: 22% to 27%


## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

With the sealed evaluation done, the frozen model scores the **latest decision date the release supports:
2026-06-22** (features 06-08 → 06-21). Its outcome lies beyond the data, which is exactly an editor's
situation on a Monday morning. Routing follows the ML-10 playbook, and the two pre-release triggers run
first.

In [10]:
LIVE = pd.Timestamp('2026-06-22')
live_con = sq.open_warehouse(sq.months_needed([LIVE]), HF_TOKEN)
live = sq.build_frame(live_con, LIVE)
prev = sealed_frames[sealed_frames['decision_date'] == SEALED_DATES[-1]]
live['p_decline'] = final_model.predict_proba(live[sq.CONTRACT_FEATURES])[:, 1]
live['lost_weekly_impressions'] = sq.lost_impressions(live)
live['expected_loss'] = sq.queue_score(live['p_decline'], live)
live = live.join(sq.route(live))
live['confidence'] = pd.cut(live['p_decline'], [0, 0.4, 0.6, 1.01], labels=['low', 'medium', 'high'], right=False).astype(str)
live_queue = live[live['action'] != 'NO_ACTION'].sort_values('expected_loss', ascending=False).reset_index(drop=True)
live_queue.insert(0, 'rank', np.arange(1, len(live_queue) + 1))
live_queue.to_csv(OUT_DIR / 'live_queue_2026-06-22.csv', index=False)          # pseudonymous IDs: gitignored

drift = {c: sq.psi(prev[c], live[c]) for c in sq.CONTRACT_FEATURES if c != 'age_days_at_T'}
site_share = (live_queue.head(50)['action'] == 'SITE_LEVEL_CHECK').mean()
print(f"{LIVE.date()}: {len(live):,} pages scored, {len(live_queue):,} in the queue | "
      f"T1 max PSI vs previous cycle {max(drift.values()):.3f} ({'FIRES' if max(drift.values()) > 0.25 else 'quiet'}), "
      f"T2 site-level share of top 50 {site_share:.2f} ({'FIRES' if site_share > 0.5 else 'quiet'})")
mix = live_queue.head(50).groupby(['action', 'reason_code']).agg(pages=('rank', 'size'), expected_loss=('expected_loss', 'sum'))
print('\nTop-50 by action:'); print(mix.to_string(), '\n')
top10 = live_queue.head(10)[['rank', 'action', 'reason_code', 'confidence', 'p_decline', 'lost_weekly_impressions',
                             'expected_loss', 'imp_pre', 'pos_pre', 'age_days_at_T']]
top10.round({'p_decline': 2, 'expected_loss': 0, 'pos_pre': 1}).set_index('rank')

2026-06-22: 76,094 pages scored, 46,709 in the queue | T1 max PSI vs previous cycle 0.049 (quiet), T2 site-level share of top 50 0.24 (quiet)

Top-50 by action:
                                        pages  expected_loss
action           reason_code                                
MONITOR          NEW_PAGE_SETTLING          2    9183.986990
                 SPIKE_FADE                 2  154518.492244
REFRESH_REVIEW   SLIDING_WITH_DEMAND       30  216994.375868
SITE_LEVEL_CHECK SITE_WIDE_SLIDE           12  142864.406686
TECHNICAL_CHECK  COLLAPSE_TO_NEAR_ZERO      4   55148.911232 



,action,reason_code,confidence,p_decline,lost_weekly_impressions,expected_loss,imp_pre,pos_pre,age_days_at_T
rank,,,,,,,,,
1,MONITOR,SPIKE_FADE,medium,0.56,269153.0,150680.0,479085.0,6.4,550
2,REFRESH_REVIEW,SLIDING_WITH_DEMAND,medium,0.49,58562.0,28885.0,183018.0,5.6,36
3,SITE_LEVEL_CHECK,SITE_WIDE_SLIDE,high,0.70,39174.0,27320.0,63596.0,17.5,276
4,TECHNICAL_CHECK,COLLAPSE_TO_NEAR_ZERO,high,0.95,28781.0,27270.0,31345.0,2.0,22
5,SITE_LEVEL_CHECK,SITE_WIDE_SLIDE,high,0.92,25107.0,23170.0,28335.0,10.9,152
6,SITE_LEVEL_CHECK,SITE_WIDE_SLIDE,high,0.61,36754.0,22241.0,126516.0,1.7,15
7,SITE_LEVEL_CHECK,SITE_WIDE_SLIDE,high,0.60,36754.0,22234.0,126516.0,1.7,16
8,REFRESH_REVIEW,SLIDING_WITH_DEMAND,medium,0.55,37412.0,20411.0,91508.0,7.1,108
9,REFRESH_REVIEW,SLIDING_WITH_DEMAND,high,0.87,13046.0,11349.0,16416.0,9.1,160


**The playbook, applied to 2026-06-22** (features 06-08 → 06-21; the outcome isn't in the data yet).
- **Scale:** 76,094 pages scored; the 46,709 that lost impressions before T enter the queue.
- **Release check:** the pre-release triggers are quiet (T1 PSI 0.049, T2 site-level share 0.24), so the queue would be released.
- **Top 50 by action:** 30 `REFRESH_REVIEW`, 12 `SITE_LEVEL_CHECK`, 4 `TECHNICAL_CHECK` and 4 `MONITOR` (2 spikes, 2 new pages).

**Ranked recommendations for the editor:**
1. **Work the 30 refresh reviews in score order.** Each is a large page whose loss is likely to persist (high or medium band). Rank 2 (183k impressions, −58.6k a week) and ranks 8–10 lead.
2. **Send the 12 site-level rows to whoever owns those sites first** (ranks 3 and 5–7 among them). When a whole site slides, a page rewrite is the wrong first move.
3. **Technical check for the 4 collapses.** Rank 4, for example, lost 28.8k a week to near zero and is 95% likely to stay down: check indexing, redirects and canonicals.
4. **Hold the spike and new-page rows for one cycle.** Rank 1 is the largest page in the queue (479k impressions), but one day carried its window. Re-score it next cycle before acting.
5. **Dedupe before review.** Ranks 6 and 7 carry identical impression series under different records, likely the same Search Console page.

**Confidence bands are calibrated on the backtest:** high-band picks persisted 83% of the time, medium 64% and low 46%.

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

All the numbers the paper shows go into one receipts file, `work/outputs/capstone_receipts.json`.
`work/scripts/build_paper.py` then renders `docs/index.html` from that file and the ML-09/ML-10 receipts.
The charts are inline SVG with hover tooltips and a table view, so the page can't drift from this notebook.

In [11]:
receipts = {
    'meta': {'release': 'FlyRank/internship-warehouse v20260703', 'dev_dates': [str(d.date()) for d in DEV_DATES],
             'sealed_dates': [str(d.date()) for d in SEALED_DATES], 'live_date': str(LIVE.date()), 'seed': SEED,
             'versions': {'python': sys.version.split()[0], 'pandas': pd.__version__, 'scikit-learn': sklearn.__version__,
                          'duckdb': importlib.import_module('duckdb').__version__}},
    'data': {'rows': len(frames), 'pages': int(frames.content_hash_id.nunique()), 'clients': int(frames.client_hash_id.nunique()),
             'dropped_incomplete': int((~frames.label_ok).sum()),
             'per_date': data_table.reset_index().assign(decision_date=lambda d: d.decision_date.dt.date.astype(str)).to_dict('records')},
    'signals': {k: v.reset_index().astype({v.index.name: str}).to_dict('records') for k, v in signals.items()},
    'backtest': {'queues': int(cells_robust.groupby(['decision_date', 'fold']).ngroups), 'results': results.to_dict('index'),
                 'bootstrap': {m: list(v) for m, v in boot.items()},
                 'per_date': per_date.reset_index().assign(decision_date=lambda d: d.decision_date.dt.date.astype(str)).to_dict('records'),
                 'contract_label': contract.to_dict('index'),
                 'routes': route_table.reset_index().to_dict('records')},
    'sealed': sealed_result,
    'live': {'pages': len(live), 'queue': len(live_queue), 'psi_max': round(max(drift.values()), 4), 'site_share_top50': round(float(site_share), 4),
             'mix_top50': mix.reset_index().round(1).to_dict('records'),
             'top10': top10.round(4).astype(str).to_dict('records')},
}
(OUT_DIR / 'capstone_receipts.json').write_text(json.dumps(receipts, indent=2, default=lambda o: o.item() if hasattr(o, 'item') else str(o)))
print('wrote', (OUT_DIR / 'capstone_receipts.json').relative_to(ROOT))

builder = ROOT / 'work' / 'scripts' / 'build_paper.py'
if builder.exists():
    run = subprocess.run([sys.executable, str(builder)], capture_output=True, text=True, cwd=ROOT)
    print(run.stdout.strip() or run.stderr.strip())
    assert run.returncode == 0, run.stderr

wrote work\outputs\capstone_receipts.json


wrote docs\index.html (66,777 bytes)


## ML-12 — demo, social cut and employer summary

### 5-minute demo outline
1. **0:00–0:30 · The decision.** 50 review slots, about 80,000 eligible pages every two weeks. The obvious answer: "review the biggest drops" (the ML-07 rule).
2. **0:30–1:30 · The tension.** In ML-08 a model hit 0.96 precision, but on tiny pages. Show the precision-vs-visibility frontier (Figure 1): the most "accurate" ranker covers about 9% of the visibility at stake.
3. **1:30–2:30 · The fix and the test.** The rule is a persistence forecast of the loss, so weight it by the learned probability that the drop persists. Explain the rolling-origin × client-grouped design (40 queues, trained only on the past, scored on unseen sites) and the whole-date bootstrap.
4. **2:30–3:30 · Results.**
   - Development: +0.054 precision (95% CI +0.030 to +0.079) at equal visibility.
   - Sealed June: the precision lead replicated, and visibility came in slightly under the rule. Say so.
   - The regime: base rate 0.14 → 0.59.
5. **3:30–4:30 · Live queue walk-through (2026-06-22).** Refresh vs site-level vs technical vs monitor, the confidence bands, and the trigger replay that would have held the queue at the April portfolio drop.
6. **4:30–5:00 · Limits and next steps.** No causal claim; next come a site-relative label, a longer backtest, and an experiment on review order.

### Social-post cut
> **Which pages should an editor review first when search visibility drops?** I tested it on 79M rows of real, pseudonymized Search Console data (FlyRank ML Internship).
>
> The surprise: the most *accurate* ranker was the least useful. Its picks really did decline (83% of the time) but were tiny, covering about 9% of the visibility at stake.
>
> Weighting a plain "biggest drop" rule by the learned chance that the drop persists raised precision@50 from 0.64 to 0.69 at the same visibility. That held across 40 review queues trained only on the past and scored on unseen sites, and the precision gain held in a sealed month.
>
> Decision support, not magic: it ranks human attention, routes site-wide and technical problems away from rewrites, and makes no causal claim.
>
> Paper + code: https://wingmanoo7.github.io/ML-INTER/

### Employer-facing summary (3 sentences)
I built a decision-support ranking system that tells content editors which pages losing search visibility to review first, on a real 79-million-row pseudonymized Search Console warehouse (FlyRank ML Internship). I evaluated it the way it would be used: trained only on past dates and tested on unseen client sites across 40 review queues plus a sealed month. A probability-weighted "lost impressions" score beat a transparent rule baseline on precision (+0.054, 95% CI +0.030 to +0.079) without losing visibility coverage in development, while the most "accurate" model would have pointed editors at tiny pages. It ships as a tested Python module (21 unit tests), executed notebooks with metric receipts, an action playbook with drift triggers, and a deployed research paper.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [x] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [x] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
